# DSN AI Hackathon 2026 — Geospatial Track
## Mapping Nigeria's Digital Divide

> **Base dataset:** Nigeria Ookla GeoPackage (broadband speed tiles)  
> **Supporting layers:** LGA & State boundaries, Health facilities, Cell towers, Relative Wealth Index  
> **Goal:** Identify connectivity gaps, overlay infrastructure and socioeconomic data, produce actionable insights.

---

## 0 · Imports & configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import fiona
import folium
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
# Files committed to the repo use a relative path from /notebooks/
# Files NOT yet in the repo use their full local path (update before sharing)
ROOT = Path('..').resolve()
DATA = ROOT / 'Data'

# Committed
OOKLA_GPKG     = DATA / 'Nigeria Ookla.gpkg'
LGA_SHP        = DATA / 'NGA_LGA_Boundaries'     / 'grid3_nga_boundary_vacclgas.shp'
STATE_SHP      = DATA / 'NGA_State_Boundaries_V2' / 'grid3_nga_boundary_vaccstates.shp'
HEALTH_SHP     = DATA / 'GRID3_NGA_health_facilities' / 'GRID3_NGA_health_facilities_v2_0.shp'
CELL_CSV       = DATA / 'cell_towers.csv'
WEALTH_CSV     = DATA / 'nga_relative_wealth_index.csv'

# NOT committed — large files; update path if running on a different machine
SCHOOLS_SHP    = DATA / 'Schools_in_Nigeria' / 'grid3_nga_poi_school.shp'
POP_RASTER     = DATA / 'nga_pop_2020_CN_100m.tif'

print('All paths resolved. ROOT =', ROOT)

## 1 · Ookla GeoPackage — layer inventory

In [ ]:
layers = fiona.listlayers(OOKLA_GPKG)
print(f'Layers in GeoPackage: {layers}')

## 2 · Load Ookla broadband layer

In [ ]:
ookla = gpd.read_file(OOKLA_GPKG, layer=layers[0])
print(f'Shape : {ookla.shape}')
print(f'CRS   : {ookla.crs}')
print(f'Columns: {list(ookla.columns)}')
ookla.head(3)

## 3 · Ookla — basic statistics

In [ ]:
# Identify the speed columns (Mbps download / upload) and test counts
num_cols = ookla.select_dtypes(include='number').columns.tolist()
print('Numeric columns:', num_cols)
ookla[num_cols].describe().round(2)

## 4 · Load administrative boundaries

In [ ]:
states = gpd.read_file(STATE_SHP)
lgas   = gpd.read_file(LGA_SHP)

print(f'States : {len(states)} features  | CRS: {states.crs}')
print(f'LGAs   : {len(lgas)} features  | CRS: {lgas.crs}')
print(f'State columns : {list(states.columns)}')
print(f'LGA columns   : {list(lgas.columns)}')

## 5 · Load health facilities

In [ ]:
health = gpd.read_file(HEALTH_SHP)
print(f'Health facilities : {len(health)} features | CRS: {health.crs}')
print(f'Columns : {list(health.columns)}')
health.head(3)

## 6 · Load cell towers (OpenCellID format)

In [ ]:
# OpenCellID CSV has no header row
CT_COLS = ['radio','mcc','net','area','cell','unit','lon','lat',
           'range','samples','changeable','created','updated','avg_signal']

towers = pd.read_csv(CELL_CSV, header=None, names=CT_COLS)
towers_gdf = gpd.GeoDataFrame(
    towers,
    geometry=gpd.points_from_xy(towers['lon'], towers['lat']),
    crs='EPSG:4326'
)

print(f'Cell towers : {len(towers_gdf):,} records')
print(f'Radio types : {towers_gdf["radio"].value_counts().to_dict()}')
towers_gdf[['radio','lon','lat','range','samples']].head(3)

## 7 · Load Relative Wealth Index

In [ ]:
wealth = pd.read_csv(WEALTH_CSV)
wealth_gdf = gpd.GeoDataFrame(
    wealth,
    geometry=gpd.points_from_xy(wealth['longitude'], wealth['latitude']),
    crs='EPSG:4326'
)

print(f'Wealth index points : {len(wealth_gdf):,}')
print(f'RWI range  : {wealth["rwi"].min():.3f} → {wealth["rwi"].max():.3f}')
print(f'RWI mean   : {wealth["rwi"].mean():.3f}')
wealth.describe().round(3)

## 8 · Reproject everything to a common CRS (EPSG:4326)

All layers will be reprojected to WGS-84 (EPSG:4326) for consistent spatial operations.

In [ ]:
WGS84 = 'EPSG:4326'

ookla  = ookla.to_crs(WGS84)
states = states.to_crs(WGS84)
lgas   = lgas.to_crs(WGS84)
health = health.to_crs(WGS84)
# towers_gdf and wealth_gdf are already in WGS84

print('All layers → EPSG:4326 ✓')

## 9 · Map 1 — Ookla download speed tiles (static)

In [ ]:
# Detect download speed column (Ookla uses avg_d_kbps or similar)
dl_col = [c for c in ookla.columns if 'd_kbps' in c.lower() or 'download' in c.lower()]
dl_col = dl_col[0] if dl_col else num_cols[0]
print(f'Using column for download speed: {dl_col}')

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Base — state boundaries
states.plot(ax=ax, facecolor='#f5f5f0', edgecolor='#888', linewidth=0.8, zorder=1)

# Ookla tiles coloured by download speed
ookla.plot(
    column=dl_col,
    ax=ax,
    cmap='RdYlGn',
    scheme='quantiles',
    k=5,
    linewidth=0,
    alpha=0.8,
    legend=True,
    legend_kwds={'title': 'Download speed (kbps)', 'loc': 'lower right', 'fontsize': 8},
    zorder=2
)

states.plot(ax=ax, facecolor='none', edgecolor='#333', linewidth=1.0, zorder=3)

ax.set_title('Nigeria — Ookla Broadband Download Speed by Tile', fontsize=14, pad=12)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('ookla_download_speed.png', dpi=150, bbox_inches='tight')
plt.show()

## 10 · Map 2 — Cell tower distribution by radio type

In [ ]:
radio_colors = {'GSM': '#e74c3c', 'UMTS': '#f39c12', 'LTE': '#2ecc71', 'NR': '#3498db'}

fig, ax = plt.subplots(1, 1, figsize=(12, 10))
states.plot(ax=ax, facecolor='#f5f5f0', edgecolor='#aaa', linewidth=0.8, zorder=1)

for radio, grp in towers_gdf.groupby('radio'):
    color = radio_colors.get(radio, '#999')
    grp.plot(ax=ax, color=color, markersize=1.5, alpha=0.5, label=radio, zorder=2)

states.plot(ax=ax, facecolor='none', edgecolor='#333', linewidth=1.0, zorder=3)

ax.legend(title='Radio type', markerscale=4, fontsize=9, loc='lower right')
ax.set_title('Nigeria — Cell Tower Distribution by Radio Type', fontsize=14, pad=12)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('cell_towers_by_radio.png', dpi=150, bbox_inches='tight')
plt.show()

## 11 · Map 3 — Relative Wealth Index

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
states.plot(ax=ax, facecolor='#f5f5f0', edgecolor='#aaa', linewidth=0.8, zorder=1)

wealth_gdf.plot(
    column='rwi',
    ax=ax,
    cmap='coolwarm',
    markersize=1.5,
    alpha=0.6,
    legend=True,
    legend_kwds={'label': 'Relative Wealth Index', 'shrink': 0.6},
    zorder=2
)

states.plot(ax=ax, facecolor='none', edgecolor='#333', linewidth=1.0, zorder=3)

ax.set_title('Nigeria — Relative Wealth Index (Meta / CIESIN)', fontsize=14, pad=12)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('relative_wealth_index.png', dpi=150, bbox_inches='tight')
plt.show()

## 12 · Map 4 — Health facilities overlaid on state boundaries

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
states.plot(ax=ax, facecolor='#eaf3fb', edgecolor='#666', linewidth=0.8, zorder=1)
health.plot(ax=ax, color='#c0392b', markersize=1.5, alpha=0.5, zorder=2)
states.plot(ax=ax, facecolor='none', edgecolor='#333', linewidth=1.0, zorder=3)

ax.set_title(f'Nigeria — Health Facilities ({len(health):,} sites)', fontsize=14, pad=12)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('health_facilities.png', dpi=150, bbox_inches='tight')
plt.show()

## 13 · Interactive map — Ookla tiles + state boundaries (Folium)

Rendered as an HTML widget; click any tile to see its speed values.

In [ ]:
# Centroid of Nigeria
nga_center = [9.0820, 8.6753]

m = folium.Map(location=nga_center, zoom_start=6, tiles='CartoDB positron')

# State outlines
folium.GeoJson(
    states,
    name='States',
    style_function=lambda x: {'fillColor': 'none', 'color': '#333', 'weight': 1.2}
).add_to(m)

# Ookla tiles — sample to keep the map responsive
ookla_sample = ookla.sample(min(5000, len(ookla)), random_state=42)

vmin = ookla_sample[dl_col].quantile(0.05)
vmax = ookla_sample[dl_col].quantile(0.95)
cmap = plt.cm.RdYlGn

def speed_color(val):
    norm = (val - vmin) / (vmax - vmin + 1e-9)
    r, g, b, _ = cmap(np.clip(norm, 0, 1))
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

folium.GeoJson(
    ookla_sample[[dl_col, 'geometry']],
    name='Ookla speed',
    style_function=lambda feat: {
        'fillColor': speed_color(feat['properties'][dl_col]),
        'color': 'none',
        'fillOpacity': 0.75,
        'weight': 0
    },
    tooltip=folium.GeoJsonTooltip(fields=[dl_col], aliases=['Download (kbps)'])
).add_to(m)

folium.LayerControl().add_to(m)
m.save('../outputs/ookla_interactive.html')
m

---
## Next steps (Stage 2)
- Spatial join: aggregate Ookla speed tiles to LGA / State level
- Correlation analysis: broadband speed × wealth index
- Coverage gap analysis: LGAs with low speed AND low cell-tower density
- School & health facility connectivity proximity analysis
- Population-weighted connectivity score